In [ ]:
#r "BoSSSpad/BoSSSpad.dll"
using System;
using System.Collections.Generic;
using System.Linq;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Foundation;
using BoSSS.Foundation.XDG;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.AdvancedSolvers;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.XNSE_Solver;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
Init();

Error: System.ApplicationException: Already called.
   at BoSSS.Application.BoSSSpad.BoSSSshell.InitTraceFile() in D:\BoSSS-experimental\public\src\L4-application\BoSSSpad\BoSSSshell.cs:line 217
   at BoSSS.Application.BoSSSpad.BoSSSshell.Init() in D:\BoSSS-experimental\public\src\L4-application\BoSSSpad\BoSSSshell.cs:line 104
   at Submission#11.<<Initialize>>d__0.MoveNext()
--- End of stack trace from previous location ---
   at Microsoft.CodeAnalysis.Scripting.ScriptExecutionState.RunSubmissionsAsync[TResult](ImmutableArray`1 precedingExecutors, Func`2 currentExecutor, StrongBox`1 exceptionHolderOpt, Func`2 catchExceptionOpt, CancellationToken cancellationToken)

In [ ]:
var myBatch = ExecutionQueues[1];
myBatch

RuntimeLocation,AdditionalEnvironmentVars,DeploymentBaseDirectory,DeployRuntime,Name,DotnetRuntime,Username,NumOfAdditionalServiceCores,NumOfAdditionalServiceCoresMPISerial,NumOfServiceCoresPerMPIprocess,ServerName,ComputeNodes,DefaultJobPriority,SingleNode,AllowedDatabasesPaths
win\amd64,[ ],\\dc3\userspace\smuda\hpccluster\binaries,True,FDY-WindowsHPC,dotnet,FDY\smuda,0,0,0,DC3,<null>,Normal,True,[ \\dc3\userspace\smuda\hpccluster\ ]


In [ ]:
BoSSSshell.WorkflowMgm.Init("RisingBubble2D", myBatch);

In [ ]:
BoSSSshell.WorkflowMgm.SetNameBasedSessionJobControlCorrelation();

In [ ]:
using BoSSS.Solution.NSECommon;
using BoSSS.Solution.XNSECommon;
using BoSSS.Solution.Timestepping;
using BoSSS.Solution.XdgTimestepping;
using BoSSS.Solution.LevelSetTools;
using BoSSS.Solution.LevelSetTools.SolverWithLevelSetUpdater;
using BoSSS.Application.XNSE_Solver.Logging;
using BoSSS.Application.XNSE_Solver.PhysicalBasedTestcases;

## Sessions

In [ ]:
bool restartRun = true;

In [ ]:
var sessions = BoSSSshell.WorkflowMgm.Sessions;
sessions

#0: RisingBubble2D	RB2D_fullDomain_20x40AMR0_k3_testcase1_withUpdatedInterfaceCheck_restart3*	09/02/2024 11:49:28	60901f69...
#1: RisingBubble2D	RB2D_fullDomain_20x40AMR0_k3_ReInit100_testcase1	08/30/2024 09:22:51	44e79500...
#2: RisingBubble2D	RB2D_fullDomain_20x40AMR0_k3_Picard_FastMarching_ReInit50_testcase1	08/30/2024 09:35:54	46f02668...
#3: RisingBubble2D	RB2D_fullDomain_20x40AMR0_k3_testcase1_withUpdatedInterfaceCheck_restart2*	08/29/2024 10:31:41	d3fbafe6...
#4: RisingBubble2D	RB2D_fullDomain_20x40AMR0_k3_testcase1_withPreEnforcerActive_restart1*	08/28/2024 15:08:37	d55fdec7...
#5: RisingBubble2D	RB2D_fullDomain_20x40AMR0_k3_testcase1*	08/27/2024 14:57:24	d9c4a861...


In [ ]:
//sessions.Pick(0).Delete(true)

In [ ]:
//sessions.Pick(0).Delete(true); //.Export().WithSupersampling(2).Do()

In [ ]:
var restartSession = sessions.Pick(3);
restartSession

RisingBubble2D	RB2D_fullDomain_20x40AMR0_k3_testcase1_withUpdatedInterfaceCheck_restart2*	08/29/2024 10:31:41	d3fbafe6...

In [ ]:
//restartSession.Timesteps.Skip(2).Export().WithSupersampling(2).Do()

In [ ]:
string restartName = "RB2D_fullDomain_20x40AMR0_k3_testcase1_OneStepGaussAndStokes_restart3";

## Grid Creation

In [ ]:
bool fullDomain = true;

In [ ]:
//int[] Resolutions = new int[] { 10, 20, 40, 80 };
int[] Resolutions = new int[] { 20 };
IGridInfo[] Grids = new IGridInfo[Resolutions.Length];

for(int i = 0; i < Resolutions.Length; i++) {
    int Res = Resolutions[i];
    string GridName = $"RisingBubble2D_{Res}x{2*Res}";
    GridName = fullDomain ? GridName + "_fullDomain" : "_halfDomain";

    IGridInfo cachedGrid = wmg.Grids.FirstOrDefault(grid => grid.Name == GridName);
    cachedGrid = null;
    if(cachedGrid == null) {
        
        // must create new Grid
        double[] xNodes = fullDomain ? GenericBlas.Linspace(0, 1.0, Res + 1) : GenericBlas.Linspace(0.5, 1.0, Res + 1);
        double[] yNodes = GenericBlas.Linspace(0, 2.0, Res*2 + 1);
        
        var grd = Grid2D.Cartesian2DGrid(xNodes, yNodes);
        grd.Name = GridName;
        
        grd.DefineEdgeTags(delegate(Vector X) {
            string ret = null;

            if (fullDomain) {
                if(X.x.Abs() <= 1e-8)
                ret = IncompressibleBcType.FreeSlip.ToString();
            } else {
                if((X.x - 0.5).Abs() <= 1e-8)
                ret = IncompressibleBcType.SlipSymmetry.ToString();
            }
            if((X.x - 1.0).Abs() <= 1e-8)
                ret = IncompressibleBcType.FreeSlip.ToString();
            if((X.y).Abs() <= 1e-8)         // lower wall
                ret = IncompressibleBcType.Wall.ToString();
            if((X.y - 2.0).Abs() <= 1e-8)   // upper wall
                ret = IncompressibleBcType.Wall.ToString();

            return ret;
        });    
        
        // grd.AddPredefinedPartitioning("FourProcSplit_fullDomain", delegate (double[] X) {
        //     int rank;
        //     double x = X[0]; double y = X[1];
        //     if (x < 0.5 && y < 0.5)
        //         rank = 0;
        //     else if (x >= 0.5 && y < 0.5)
        //         rank = 1;
        //     else if (x < 0.5 && y >= 0.5)
        //         rank = 2;
        //     else 
        //         rank = 3;

        //     return rank;
        // });
        
        Grids[i] = wmg.SaveGrid(grd);
        
    } else {
        //Console.WriteLine($"type: {cachedGrid.GetType()}, is IGridInfo? {cachedGrid is IGridInfo}");
        Console.WriteLine("Grid already found in database - identifid by name " + GridName);
        Grids[i] = cachedGrid;
    }
    
}

## Initial Values

In [ ]:
var PhiFunc = new Formula(
    "Phi",
    false,
    "using ilPSP.Utils; " + 
    "double dropRadius = 0.25;" + 
    "double Phi(double[] X) { " + 
    "    return Math.Sqrt((X[0] - 0.5).Pow2() + (X[1] - 0.5).Pow2()) - dropRadius; " + 
    "}");

In [ ]:
var GravityValue = new Formula(
    "grav",
    false,
    "using ilPSP.Utils; " + 
    "double grav(double[] X) { return -0.98; }");

## Physical Settings

In [ ]:
string testcase = "testcase1";

In [ ]:
double rhoA = 1.0;      // bubble
double rhoB = 1.0;   // surrounding fluid
double muA = 1.0;
double muB = 1.0;
double sigma = 0.0;

switch(testcase) {
    case "testcase1": {
        rhoA = 100.0;
        rhoB = 1000.0;
        muA = 1.0;
        muB = 10.0;
        sigma = 24.5;

        break;
    }
    case "testcase2": {
        rhoA = 1.0;
        rhoB = 1000.0;
        muA = 0.1;
        muB = 10.0;
        sigma = 1.96;
        
        break;
    }
}


In [ ]:
int k = 3;
int AMRlevel = 0;
double hmin = (1.0 / (double)Resolutions[0]) / ((double)AMRlevel + 1);
double dt_sigma = BoSSS.Solution.XNSECommon.XNSEUtils.GetCapillaryTimeStep(rhoA, rhoB, sigma, hmin, k+1);
dt_sigma

0.0026731502275068254

In [ ]:
double dt = 0.002;

In [ ]:
int numTs = (int)(3.0/dt);
numTs

1500

## Control settings

In [ ]:
List<XNSE_Control> Controls = new List<XNSE_Control>();

In [ ]:
restartSession.Timesteps.Last()

 { Time-step: 1102; Physical time: 2.201999999999979s; Fields: Phi, PhiDG, VelocityX, VelocityY, Pressure, Residual-MomentumX, Residual-MomentumY, Residual-ContiEq, Velocity0X_Mean, Velocity0Y_Mean, GravityY#A, GravityY#B, VelocityX@Phi, VelocityY@Phi, MPIrank, CutCells; Name:  }

In [ ]:
int restartTS = restartSession.Timesteps.Last().TimeStepNumber.MajorNumber - 1;
restartTS

1101

In [ ]:
for (int idx = 0; idx < Grids.Length; idx++) {

    XNSE_Control C = new XNSE_Control();

    C.SetDGdegree(k);
    C.FieldOptions.Add(VariableNames.GravityY + "#A", new FieldOpts() {
        SaveToDB = FieldOpts.SaveToDBOpt.TRUE
    });
    C.FieldOptions.Add(VariableNames.GravityY + "#B", new FieldOpts() {
        SaveToDB = FieldOpts.SaveToDBOpt.TRUE
    });

    // physical parameters
    C.PhysicalParameters.rho_A = rhoA;
    C.PhysicalParameters.mu_A = muA;
    
    C.PhysicalParameters.rho_B = rhoB;
    C.PhysicalParameters.mu_B = muB;
    
    C.PhysicalParameters.Sigma = sigma;
    
    C.PhysicalParameters.IncludeConvection = true;

    
    if (restartRun) {
        C.RestartInfo = new Tuple<Guid, BoSSS.Foundation.IO.TimestepNumber>(restartSession.ID, restartTS);
    } else {
        // set grid
        C.SetGrid(Grids[idx]);
    
        // initial conditions   
        C.AddInitialValue("Phi", PhiFunc);   
         
        C.AddInitialValue("GravityY#A", GravityValue);
        C.AddInitialValue("GravityY#B", GravityValue);
    }
    
    C.AdvancedDiscretizationOptions.SST_isotropicMode = SurfaceStressTensor_IsotropicMode.LaplaceBeltrami_ContactLine;
    C.LSContiProjectionMethod = ContinuityProjectionOption.ConstrainedDG;
 
    // C.ReInitPeriod = 50;

    // C.SkipSolveAndEvaluateResidual = true;
    C.CutCellQuadratureType = BoSSS.Foundation.XDG.XQuadFactoryHelper.MomentFittingVariants.OneStepGaussAndStokes;

    // C.NonLinearSolver.SolverCode = NonLinearSolverCode.Picard;
    // C.NonLinearSolver.ConvergenceCriterion = 1e-9;
    C.NonLinearSolver.MaxSolverIterations = 50;

    // C.LinearSolver = LinearSolverCode.exp_Kcycle_schwarz.GetConfig();


    C.TimesteppingMode = AppControl._TimesteppingMode.Transient;
    C.Timestepper_LevelSetHandling = LevelSetHandling.Coupled_Once;
    C.Option_LevelSetEvolution = LevelSetEvolution.StokesExtension;
    C.TimeSteppingScheme = TimeSteppingScheme.BDF3;
    C.dtFixed = dt;
    C.NoOfTimesteps = numTs;

    C.saveperiod = 1;

    if (AMRlevel > 0) {
        C.AdaptiveMeshRefinement = true;
        C.activeAMRlevelIndicators.Add(new AMRonNarrowband() { maxRefinementLevel = AMRlevel });
        C.AMR_startUpSweeps = AMRlevel;
    }

    C.PostprocessingModules.Add(new RisingBubble2DBenchmarkQuantities());
    // C.TracingNamespaces = \"BoSSS.Solution.AdvancedSolvers\";
   
    if (restartRun) {
        C.SessionName = restartName;
    } else {
        int res = Resolutions[idx];
        C.SessionName = $"RB2D_fullDomain_{res}x{2*res}AMR{AMRlevel}_k{k}_Picard_FastMarching_ReInit50_{testcase}";
        //C.SessionName = $"RB3D_quarterDomain_{res}x{res}x{2*res}AMR{AMRlevel}_k{k}_{testcase}_setttingsCheck";
    }
    
    // C.GridPartType = GridPartType.Predefined;
    // C.GridPartOptions = "FourProcSplit_fullDomain";
    
    Controls.Add(C);
}

In [ ]:
Controls.Select(C => C.SessionName)

index,value
0,RB2D_fullDomain_20x40AMR0_k3_testcase1_OneStepGaussAndStokes_restart3


## Launch Jobs

In [ ]:
foreach(var ctrl in Controls) {
    var oneJob              = ctrl.CreateJob();
    
    oneJob.NumberOfMPIProcs = 1;

    int numThreads = 2;
    oneJob.NumberOfThreads = numThreads;
    IDictionary<string, string> args = oneJob.EnvironmentVars;
    args["BOSSS_ARG_7"] = numThreads.ToString();
    args.Remove("OMP_NUM_THREADS");
    oneJob.MySetCommandLineArguments(args.Values.ToArray());

    //oneJob.UseComputeNodesExclusive = true;\n",

    //((SlurmClient)myBatch).ExecutionTime = "24:00:00";

    oneJob.Activate(myBatch);
}